<a href="https://colab.research.google.com/github/JoyeeChen/congenial-computing-machine/blob/master/Copy_of_Copy_of_Apr29FirstRunnableMoreRobustTrainingRound1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install unsloth peft datasets transformers huggingface_hub

In [ ]:
from huggingface_hub import login
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(HF_TOKEN) # Still good practice to log in
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

In [ ]:
# Progressive LoRA + Merge + KL Training Pipeline

from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import torch.nn.functional as F
import copy, os
import matplotlib.pyplot as plt


base_model = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
hf_dataset_path = "CompassioninMachineLearning/10k_Animal_CoT_data_day90_third_path_multi_turn_mixed"  # Replace with your HuggingFace dataset path
num_chunks = 5
rank_schedule = [8, 16, 32, 32, 64]
lora_alpha_scale = 8
lora_dropout = 0.05
kl_weight = 0.2
total_train_samples = 10000  # Set this to how much total training data you want
save_path = "merged_model_outputs"
os.makedirs(save_path, exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
'''import torch, gc

del merged_model
del trainer
gc.collect()
torch.cuda.empty_cache()'''

'import torch, gc\n\ndel merged_model\ndel trainer\ngc.collect()\ntorch.cuda.empty_cache()'

In [ ]:
# === LOAD TOKENIZER & BASE MODEL ===
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    load_in_4bit = load_in_4bit,
    dtype = dtype,
    max_seq_length = max_seq_length,          #  ⇠  halve flops vs. 2‑k tokens
    #low_cpu_mem_usage = True,
    trust_remote_code = True,
)
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.4.3: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Embedding(128256, 4096, padding_idx=128004)

In [ ]:
import gc
gc.collect()
import torch
torch.cuda.empty_cache()

In [ ]:
# ---------------------------------------------------------------------------
# 1. Alpaca-style prompt → tensors
# ---------------------------------------------------------------------------
import numpy as np

def build_prompt(inst: str, inp: str, resp: str) -> str:
    """Create the full Alpaca-style prompt with eos."""
    if inp and inp.strip():
        return (
            f"### Instruction:\n{inst}\n\n"
            f"### Input:\n{inp}\n\n"
            f"### Response:\n{resp}{tokenizer.eos_token}"
        )
    return (
        f"### Instruction:\n{inst}\n\n"
        f"### Response:\n{resp}{tokenizer.eos_token}"
    )

def alpaca_preprocess(batch):
    # ❗  Use the correct column names: instruction / input / response
    prompts = [
        build_prompt(i, inp, resp)
        for i, inp, resp in zip(batch["instruction"],
                                batch["input"],
                                batch["output"])
    ]

    tok = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=min(1024, tokenizer.model_max_length),
    )

    # ❗  Create a *copy* before masking so we don’t alter tokenizer output in-place
    labels = np.array(tok["input_ids"])
    labels[labels == tokenizer.pad_token_id] = -100
    tok["labels"] = labels.tolist()
    return tok

# ---------------------------------------------------------------------------
# 2. Load your dataset and make a clean train/val split
# ---------------------------------------------------------------------------
from datasets import load_dataset

data = load_dataset(hf_dataset_path)           # <-- your HF path

train_src = data["train"]
val_src   = (
    data.get("validation")                     # ❗  prefer dedicated val split
    or data.get("test")                        # fallback to test if it exists
)

if val_src is None:
    # Final fallback: split 10 % of train into validation
    split = train_src.train_test_split(test_size=0.1, seed=42)
    train_src, val_src = split["train"], split["test"]

# (Optional) subsample for quick runs
train_src = train_src.select(range(min(10_000, len(train_src))))
val_src   = val_src.select(range(min(1_000,  len(val_src))))

# ---------------------------------------------------------------------------
# 3. Tokenise
# ---------------------------------------------------------------------------
NUM_PROC   = 2
BATCH_SIZE = 200

tokenized_train = train_src.map(
    alpaca_preprocess,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROC,
    remove_columns=train_src.column_names,
    desc="tokenising-train",
)

tokenized_val = val_src.map(
    alpaca_preprocess,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROC,
    remove_columns=val_src.column_names,
    desc="tokenising-val",
)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/4.32M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/436k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

tokenising-train (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

tokenising-val (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import sys
import psutil
from IPython import get_ipython

def analyze_memory_usage(n_top=10, sort_by_type=False):
    """
    Analyze the memory usage of variables in the current IPython session.

    Parameters:
    -----------
    n_top : int
        Number of top memory-consuming variables to display
    sort_by_type : bool
        If True, summarize memory usage by variable type

    Returns:
    --------
    None (prints results)
    """
    # Get dictionary of all variables in user namespace
    user_namespace = get_ipython().user_ns

    # Filter out IPython's internal variables (starting with '_')
    user_vars = {key: value for key, value in user_namespace.items()
                if not key.startswith('_') and not callable(value)}

    # Calculate memory usage for each variable
    var_info = []
    for var_name, var_value in user_vars.items():
        try:
            var_size = sys.getsizeof(var_value)
            var_type = type(var_value).__name__
            var_info.append((var_name, var_type, var_size))
        except:
            # Skip variables that cause issues with sys.getsizeof
            continue

    # Sort by memory usage (largest first)
    var_info.sort(key=lambda x: x[2], reverse=True)

    # Print overall memory information
    process = psutil.Process()
    print(f"Total process memory usage: {process.memory_info().rss / (1024 * 1024):.2f} MB")
    print("\n----- Top Memory Variables -----")

    # Print top variables by memory usage
    print(f"{'Name':<20} {'Type':<15} {'Size (bytes)':<12} {'Size (MB)':<10}")
    print("-" * 60)

    for i, (name, var_type, size) in enumerate(var_info[:n_top]):
        size_mb = size / (1024 * 1024)
        print(f"{name:<20} {var_type:<15} {size:,} {size_mb:.4f}")

    # Summarize by type if requested
    if sort_by_type:
        print("\n----- Memory Usage by Type -----")
        type_sizes = {}
        for _, var_type, size in var_info:
            if var_type in type_sizes:
                type_sizes[var_type] += size
            else:
                type_sizes[var_type] = size

        # Sort types by total memory usage
        sorted_types = sorted(type_sizes.items(), key=lambda x: x[1], reverse=True)

        print(f"{'Type':<15} {'Total Size (bytes)':<18} {'Total Size (MB)':<12} {'Count':<5}")
        print("-" * 60)

        for var_type, total_size in sorted_types:
            count = sum(1 for item in var_info if item[1] == var_type)
            size_mb = total_size / (1024 * 1024)
            print(f"{var_type:<15} {total_size:,} {size_mb:.4f} {count:<5}")

# Example usage
# analyze_memory_usage(n_top=15, sort_by_type=True)
analyze_memory_usage(n_top=15, sort_by_type=False)

Total process memory usage: 2007.34 MB

----- Top Memory Variables -----
Name                 Type            Size (bytes) Size (MB) 
------------------------------------------------------------
Out                  dict            224 0.0002
data                 DatasetDict     208 0.0002
hf_dataset_path      str             130 0.0001
In                   list            120 0.0001
rank_schedule        list            104 0.0001
base_model           str             92 0.0001
HF_TOKEN             str             86 0.0001
os                   module          72 0.0001
userdata             module          72 0.0001
torch                module          72 0.0001
F                    module          72 0.0001
copy                 module          72 0.0001
plt                  module          72 0.0001
gc                   module          72 0.0001
np                   module          72 0.0001


In [ ]:
'''merged_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs/checkpoint-60",
    load_in_4bit = load_in_4bit,
    dtype = dtype,
    max_seq_length = max_seq_length,          #  ⇠  halve flops vs. 2‑k tokens
    trust_remote_code = True,
)
tokenizer.pad_token = tokenizer.eos_token'''

'merged_model, tokenizer = FastLanguageModel.from_pretrained(\n    model_name = "outputs/checkpoint-60",\n    load_in_4bit = load_in_4bit,\n    dtype = dtype,\n    max_seq_length = max_seq_length,          #  ⇠  halve flops vs. 2‑k tokens\n    trust_remote_code = True,\n)\ntokenizer.pad_token = tokenizer.eos_token'

In [ ]:
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10000
})

In [ ]:
#IF YOU'RE NOT RUNNING THIS ON AN A100: MUST CHANGE REFERENCES TO BFLOAT16 BELOW TO FLOAT16!!

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback, AutoModelForCausalLM, DataCollatorForSeq2Seq
from transformers.trainer_callback import ProgressCallback
import json, os, torch, gc, datetime, time, copy
from trl import SFTTrainer
from unsloth import is_bfloat16_supported, train_on_responses_only
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Missing initialization for these variables
# Assuming model, tokenizer, tokenized_train, tokenized_val are defined before this code
# Add these lines where appropriate in your full script:
# model = AutoModelForCausalLM.from_pretrained("your_base_model")
# tokenizer = AutoTokenizer.from_pretrained("your_base_model")
# tokenized_train = ... # Your training dataset
# tokenized_val = ... # Your validation dataset

# Ensure model parameters have requires_grad=True
#for param in model.parameters():
#    param.requires_grad = True

# Store original model to reuse in subsequent rounds
#base_model = copy.deepcopy(model)

rank_schedule = [8, 16, 24]
epochs_per_round = [2, 2, 2]
learning_rates = [5e-4, 2.5e-4, 1e-4]

# Dictionary to store training metrics
training_metrics = {
    "rounds": []
}

for round_idx, (rank, epochs, lr) in enumerate(zip(rank_schedule, epochs_per_round, learning_rates)):
    print(f"Starting round {round_idx+1} with rank {rank}, epochs {epochs}, lr {lr}")

    peft_config = LoraConfig(
        r=rank,
        lora_alpha=rank * 4,
        lora_dropout=0.0,
        bias="none",
        use_rslora=False,  # We support rank stabilized LoRA
        task_type=TaskType.CAUSAL_LM,  # Explicitly specify the task type
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=None  # Don't try to save any modules fully
    )

    # Make sure the model is prepared for training before applying LoRA
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)

    model = model.to(torch.bfloat16)

    # Create a custom callback to track validation loss
    class ValidationTracker(ProgressCallback):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.val_losses = []

        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics and "eval_loss" in metrics:
                self.val_losses.append((state.global_step, metrics["eval_loss"]))
                print(f"Step {state.global_step}: Validation Loss = {metrics['eval_loss']}")

    # Initialize the custom callback
    val_tracker = ValidationTracker()

    # Set model to train mode explicitly
    model.train()

    # Create a fresh trainer for this round
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        dataset_num_proc=2,
        # Don't pass peft_config again as we've already applied it
        packing=False,  # Can make training 5x faster for short sequences.
        args=TrainingArguments(
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=2,
            logging_steps=10,  # Increased for less frequent logging
            optim="adamw_torch_fused",
            eval_strategy="steps",  # Fixed: eval_strategy → evaluation_strategy
            eval_steps=40,  # Evaluate less frequently for better performance
            save_steps=40,
            gradient_checkpointing=True,
            learning_rate=lr,
            warmup_steps=100,
            num_train_epochs=epochs,
            weight_decay=0.001,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            output_dir=f"outputs/round{round_idx+1}",  # Separate output dirs for each round
            report_to="none",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",  # Specify which metric to track
            greater_is_better=False,
            dataloader_num_workers=4,
            torch_compile=False,
            group_by_length=True,
            remove_unused_columns=False,
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3), val_tracker]  # Added early stopping and val_tracker
    )

    # Train the model
    train_result = trainer.train()

    # Save training metrics
    round_metrics = {
        "rank": rank,
        "epochs": epochs,
        "learning_rate": lr,
        "train_loss": train_result.training_loss,
        "validation_losses": val_tracker.val_losses
    }
    training_metrics["rounds"].append(round_metrics)

    # Save metrics to file
    with open(f"outputs/round{round_idx+1}/training_metrics.json", "w") as f:
        json.dump(round_metrics, f, indent=2)

    # Merge and save the model
    print(f"Saving merged model for round {round_idx+1}")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(f"outputs/round{round_idx+1}/merged")
    tokenizer.save_pretrained(f"outputs/round{round_idx+1}/merged")

    # Assign "model" variable to a deep copy of merged_model
    model = copy.deepcopy(merged_model)
    tokenizer = copy.deepcopy(tokenizer)

    # Clean up to free memory
    del merged_model
    gc.collect()
    torch.cuda.empty_cache()

# Save overall training metrics
with open("outputs/training_metrics.json", "w") as f:
    json.dump(training_metrics, f, indent=2)

print("Training complete!")

Starting round 1 with rank 8, epochs 2, lr 0.0005


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 2 | Total steps = 624
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 20,971,520/8,000,000,000 (0.26% trained)


  0%|          | 0/624 [00:00<?, ?it/s]

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
40,1.496000,1.741854
80,1.336300,1.690230
120,1.303400,1.733586
160,1.254300,1.840402
200,1.259300,1.864440


...

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Starting round 3 with rank 24, epochs 2, lr 0.0001


/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 2 | Total steps = 624
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 62,914,560/8,000,000,000 (0.79% trained)


  0%|          | 0/624 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss


In [ ]:
peft_config

In [ ]:
get_peft_model(model, peft_config)

In [ ]:
'''from transformers import TrainingArguments, EarlyStoppingCallback, AutoModelForCausalLM
from transformers.trainer_callback import ProgressCallback
import json, os, torch, gc, datetime, time
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.checkpoint import checkpoint_sequential

# -------- Config --------
progress_file = "plora_progress.json"
# Progressive LoRA configurations
learning_rate_schedule = [1e-5, 5e-6, 2.5e-6]
rank_schedule = [8, 16, 24]  # Progressively increasing ranks
#learning_rate_schedule = [2e-5, 1e-5, 5e-6]
lora_dropout_schedule = [0.05, 0.05, 0.08]
lora_alpha_scale = 4
num_stages = len(rank_schedule)  # Number of progressive stages
load_in_4bit = True
dtype = None

# Use full dataset
train_ds = tokenized_train
val_ds = tokenized_val

# -------- Load progress & optional overrides --------
manual_start_stage = 3   # 1-based user-specified start (if None, use the next stage after the last completed one)
resume_checkpoint = 'merged_model_round1_chunk1/'
completed_stages = []
stage_losses = []
val_losses = []

if os.path.exists(progress_file):
    with open(progress_file) as f:
        progress = json.load(f)
    manual_start_stage = progress.get("manual_start_stage")
    resume_checkpoint = progress.get("resume_checkpoint")
    completed_stages = progress.get("completed_stages", [])
    stage_losses = progress.get("stage_losses", [])
    val_losses = progress.get("val_losses", [])

# Determine which stage to start from and which checkpoint to load
if manual_start_stage is not None:
    next_stage_idx = manual_start_stage - 1  # Convert to 0-based
    print(f"Manual start_stage={manual_start_stage}, will begin at stage index {next_stage_idx}")
    # Only clear completed stages if we're starting from an earlier point
    if not completed_stages or next_stage_idx <= max(completed_stages):
        # Keep completed stages that are before our manual starting point
        completed_stages = [s for s in completed_stages if s < next_stage_idx]
        print(f"Keeping prior completed stages: {completed_stages}")
elif completed_stages:
    next_stage_idx = max(completed_stages) + 1
    if next_stage_idx >= num_stages:
        print(f"All stages already completed (0 to {num_stages-1}). Set manual_start_stage to rerun specific stages.")
        next_stage_idx = num_stages  # Will exit the loop
    else:
        print(f"Resuming with next stage {next_stage_idx} after last completed stage {max(completed_stages)}")
else:
    next_stage_idx = 0
    print("No checkpoint found; starting from scratch with stage 0")

# Determine which checkpoint to load
if resume_checkpoint:
    checkpoint = resume_checkpoint
    print(f"Loading model from override checkpoint: {checkpoint}")
elif completed_stages:
    last_completed = max(completed_stages)
    checkpoint = f"./plora_model_stage{last_completed}"
    print(f"Loading model from last completed stage: {last_completed}")
else:
    checkpoint = None
    print("No checkpoint to load, will start with base model")

# Load or initialize model
if checkpoint:
    print(f"Loading model from: {checkpoint}")
    merged_model, tokenizer = FastLanguageModel.from_pretrained(
        checkpoint,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=dtype,
        trust_remote_code=True,
        local_files_only=True
    )
else:
    merged_model = model  # initial base

# -------- Progressive LoRA Training loop --------
for stage_idx in range(num_stages):
    # Skip if we're before our next stage to train
    if stage_idx < next_stage_idx:
        print(f"Skipping stage {stage_idx+1}/{num_stages} (already completed or before manual start)")
        continue

    print(f"\n==== Training PLORA stage {stage_idx+1}/{num_stages} ====")
    stage_start_time = time.time()

    # cleanup
    for v in ('model', 'trainer'): locals().pop(v, None)
    gc.collect(); torch.cuda.empty_cache()

    # Enable flash attention if available
    os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
    os.environ['TOKENIZERS_PARALLELISM'] = 'true'

    # Try to set optimal CUDA settings
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 on Ampere

    # Progressive hyperparams for this stage
    lr = learning_rate_schedule[min(stage_idx, len(learning_rate_schedule)-1)]
    r = rank_schedule[min(stage_idx, len(rank_schedule)-1)]
    dropout = lora_dropout_schedule[min(stage_idx, len(lora_dropout_schedule)-1)]
    alpha = r * lora_alpha_scale
    print(f"LoRA rank={r}, alpha={alpha}, dropout={dropout}, lr={lr}")

    # LoRA config for this stage
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = prepare_model_for_kbit_training(merged_model)
    model = get_peft_model(model, lora_config)

    # training args with performance optimizations
    training_args = TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=1,              # Reduced to single epoch per stage for speed
        #max_steps=2,
        logging_strategy="steps",
        logging_steps=50,                # Less frequent logging for speed
        optim="adamw_torch_fused",
        eval_strategy="steps",
        eval_steps=50,                  # Less frequent evaluation
        save_steps=100,                  # Less frequent saving
        gradient_checkpointing=True,
        learning_rate=lr,
        warmup_steps=100,
        lr_scheduler_type="cosine",
        #learning_rate=lr,
        weight_decay=0.001,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir=f"outputs_stage{stage_idx}",
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        dataloader_num_workers=4,        # Parallel data loading
        torch_compile=False,
        group_by_length=True             # Reduces padding, improves throughput
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,  # Using full dataset
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=0)]
    )

    # train & eval
    os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
    trainer.train()

    # Debug: confirm learning rate is set correctly (do this AFTER training starts)
    actual_lr = trainer.optimizer.param_groups[0]['lr']
    print(f"Actual learning rate from optimizer: {actual_lr}")
    if abs(actual_lr - lr) > 1e-6:  # Using larger tolerance for floating point comparison
        print(f"WARNING: Actual learning rate {actual_lr} differs from configured learning rate {lr}")

    eval_metrics = trainer.evaluate()

    # merge & save
    merged_model = model.merge_and_unload()

    ckpt = f"./plora_model_stage{stage_idx}"
    merged_model.save_pretrained_merged(ckpt, tokenizer, save_method="merged_16bit")

    # Test generation between stages
    print("\n===== Testing text generation with current model =====")
    test_prompts = [
        "A farmers Pardiumlon ranch burnt down. He was distraught and said he had lost everthing. Who suffered here?"
    ]

    # Create generation config
    generate_kwargs = {
        "max_new_tokens": 50,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True
    }

    # Generate and print samples
    for prompt in test_prompts:
        print(f"\nPrompt: {prompt}")
        inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
        with torch.inference_mode():
            outputs = merged_model.generate(**inputs, **generate_kwargs)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Generated: {generated_text}")

    # update progress
    if stage_idx not in completed_stages:
        completed_stages.append(stage_idx)
    # Update losses, preserving historical data for already completed stages
    while len(stage_losses) <= stage_idx:
        stage_losses.append(None)
    while len(val_losses) <= stage_idx:
        val_losses.append(None)

    stage_losses[stage_idx] = trainer.state.log_history[-1].get('loss')
    val_losses[stage_idx] = eval_metrics.get('eval_loss')

    with open(progress_file, 'w') as f:
        json.dump({
            "manual_start_stage": manual_start_stage,
            "resume_checkpoint": resume_checkpoint,
            "completed_stages": sorted(completed_stages),  # Sort for clarity
            "stage_losses": stage_losses,
            "val_losses": val_losses,
            "latest_stage_completion_time": str(datetime.datetime.now())
        }, f)

    # Calculate and display stage duration
    stage_duration = time.time() - stage_start_time
    hours, remainder = divmod(stage_duration, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f"\nStage {stage_idx+1} completed in {int(hours)}h {int(minutes)}m {int(seconds)}s")

    # Clean up
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()

    # No need to reload the model between stages
    # Just clean up to avoid memory issues
    if stage_idx < num_stages - 1:
        gc.collect(); torch.cuda.empty_cache()
        print(f"Moving to next PLORA stage with existing merged model")

print("Progressive LoRA training complete!")
print(f"Completed stages: {sorted(completed_stages)}")
print(f"Training losses: {stage_losses}")
print(f"Validation losses: {val_losses}")'''

In [ ]:
trainer.optimizer.param_groups

In [ ]:
del merged_model

In [ ]:
# Take 32 examples, run a single forward pass
sample = torch.utils.data.Subset(tokenized_val, range(32))
with torch.inference_mode():
    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in sample[:].items()}
    outputs = model(**inputs)
print("Manual CE:", outputs.loss.item())

In [ ]:
eval_metrics = trainer.evaluate()

# Record average losses
avg_train_loss = sum(trainer.losses) / len(trainer.losses)
round_losses.append(avg_train_loss)
val_losses.append(eval_metrics["eval_loss"])

# Merge & Save
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{save_path}/round_{round_idx+1}")
tokenizer.save_pretrained(f"{save_path}/round_{round_idx+1}")

In [ ]:
val_losses

In [ ]:
# === PLOT LOSS CURVES ===
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_chunks+1), round_losses, marker='o', label="Train Loss")
plt.plot(range(1, num_chunks+1), val_losses, marker='s', label="Validation Loss")
plt.title("Training and Validation Loss per Chunk")
plt.xlabel("Chunk Number")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

print("\nAll chunks processed and merged model saved.")


In [ ]:
prompt = (
"Roughly how much petrol would it take for a 2002 AU Falcon to do "
"a Cannonball Run from Perth to Sydney? Think step by step."
)
gen = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)
print(gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)[0]["generated_text"])

In [ ]:
merged_model

In [ ]:
merged_model

In [ ]:
#merged_model = merged_model.merge_and_unload()
model.save_pretrained_merged("10k_four_fifths_animals_PLORA", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged("CompassioninMachineLearning/10k_four_fifths_animals_PLORA", tokenizer, save_method = "merged_16bit")

In [ ]:
#merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained_merged("10k_four_fifths_animals_PLORA", tokenizer, save_method = "merged_16bit",)
merged_model.push_to_hub_merged("CompassioninMachineLearning/10k_four_fifths_animals_PLORA", tokenizer, save_method = "merged_16bit")